Tools

Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.
Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a chat model. The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.

In [7]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Tell me aboout the CNN in deep learning!")
print(response)

content='## Convolutional Neural Networks (CNNs) – A Deep‑Learning Primer  \n\nConvolutional Neural Networks (CNNs, also called ConvNets) are a family of deep‑learning models that excel at extracting spatial hierarchies of features from grid‑like data (e.g., images, video frames, spectrograms, or even 1‑D signals). They have become the work‑horse behind most modern computer‑vision systems and are increasingly used in domains such as medical imaging, remote sensing, audio processing, and natural‑language tasks that can be cast as “spatial” problems.\n\nBelow is a structured overview that covers **what a CNN is, how it works, its key components, typical architectures, training tricks, and practical tips**.\n\n---\n\n### 1. Why Convolutions?  \n\n| Property | Fully‑Connected (FC) Layer | Convolutional Layer |\n|----------|----------------------------|---------------------|\n| **Parameter count** | Grows with input size (e.g., 224×224×3 → 150\u202fk weights per neuron) | Shared weights → f

In [8]:
from langchain.tools import tool

@tool

def get_whether(location:str)->str:
    """get the wheather at the location"""
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_whether])

In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks about weather in Boston. Need to use get_whether function.', 'tool_calls': [{'id': 'fc_fd8898ae-bee1-4579-8302-b2e7b82f435a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_whether'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 129, 'total_tokens': 173, 'completion_time': 0.091540771, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006266069, 'prompt_tokens_details': None, 'queue_time': 0.392494919, 'total_time': 0.09780684}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1957c-89c2-74e3-9677-59baaa31e2d2-0' tool_calls=[{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': 'fc_fd8898ae-bee1-4579-8302-b2e7b82f435a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'in

In [ ]:
###tool excution loops

import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool


# --------------------------------
# 1. Load Environment Variables
# --------------------------------

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")


# --------------------------------
# 2. Initialize Groq Model
# --------------------------------

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)


# --------------------------------
# 3. Create First Tool
# --------------------------------

@tool
def get_weather(city: str) -> str:
    """Get the temperature of a given city in Celsius."""

    weather_data = {
        "delhi": 25,
        "mumbai": 30,
        "bangalore": 22,
        "jaipur": 28
    }

    temperature = weather_data.get(city.lower())

    if temperature is None:
        return f"Weather data not available for {city}"

    return f"The temperature in {city} is {temperature} Celsius."


# --------------------------------
# 4. Create Second Tool
# --------------------------------

@tool
def convert_to_fahrenheit(celsius: float) -> float:
    """Convert Celsius temperature into Fahrenheit."""

    fahrenheit = (celsius * 9 / 5) + 32

    return fahrenheit


# --------------------------------
# 5. Create Agent
# --------------------------------

tools = [
    get_weather,
    convert_to_fahrenheit
]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful weather assistant. "
        "Use get_weather to find the city's temperature. "
        "Then use convert_to_fahrenheit to convert it. "
        "Finally, provide both Celsius and Fahrenheit values."
    )
)


# --------------------------------
# 6. Execute Agent
# --------------------------------

if __name__ == "__main__":

    user_query = input("Ask your question: ")

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_query
                }
            ]
        }
    )

    # --------------------------------
    # 7. Print Execution History
    # --------------------------------

    print("\n========== AGENT EXECUTION ==========\n")

    for message in response["messages"]:

        print(f"Message Type: {message.type}")

        if message.type == "ai":
            print("AI Response:", message.content)

            if message.tool_calls:
                print("Tool Calls:", message.tool_calls)

        elif message.type == "tool":
            print("Tool Name:", message.name)
            print("Tool Output:", message.content)

        print("------------------------------------")

    # --------------------------------
    # 8. Print Final Answer
    # --------------------------------

    print("\n========== FINAL ANSWER ==========\n")

    print(response["messages"][-1].content)
